# MiniVLMDocEval — reproducible runner

Run this notebook **top to bottom** to reproduce every reported result. Because the six
models split across **two incompatible `transformers` versions**, reproduction is **two phases**
with a **runtime restart** between them (results persist on Drive and merge into one table):

- **Phase A** — Env A (`transformers<4.57`): SmolVLM2, InternVL3, LLaVA-OneVision, FastVLM.
- **Phase B** — Env B (`transformers@main`): Qwen3.5-0.8B, LFM2.5-VL-450M, and the Part-2 study.
- **Phase C** — figures (either env).

Set the runtime to a **GPU** (T4 works for Phase A; prefer **A100/L4** for Phase B).

## Phase A — Env A models (`transformers<4.57`)

In [ ]:
# Phase A · Setup — Env A (transformers<4.57). Clone repo + install the eval engine.
import os
REPO = '/content/MiniVLMDocEval'
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > GPU'
if not os.path.isdir(REPO):
    !git clone https://github.com/SajjadPSavoji/MiniVLMDocEval.git $REPO
%cd $REPO
!git pull --quiet && bash setup.sh

In [ ]:
# Drive — persist all outputs under OUT_DIR (survives disconnects; Phase A & B merge here).
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/MiniVLMDocEval/outputs'
os.makedirs(OUT_DIR, exist_ok=True); print('OUT_DIR ->', OUT_DIR)

In [ ]:
# Phase A · Smoke test — built-ins × datasets × 10 samples (fast end-to-end sanity).
!python scripts/smoke_test.py --score

In [ ]:
# Phase A · Efficiency ablations — fp16 speedup + generation-cap accuracy-neutrality (Table 3).
!python scripts/efficiency_ablation.py --out $OUT_DIR/ablation --n 100

In [ ]:
# Phase A · Full eval — the 3 VLMEvalKit-native models (N=1000, seed 42).
!python scripts/run_eval.py --out $OUT_DIR --n 1000 --fp16

In [ ]:
# Phase A · Full eval — FastVLM-0.5B (Env A custom wrapper).
!python scripts/run_eval.py --out $OUT_DIR --n 1000 --fp16 --models FastVLM-0.5B

---
## ⚠️ RESTART THE RUNTIME BEFORE PHASE B

Phase B needs `transformers@main`, incompatible with Phase A's pinned `<4.57`.
**Runtime → Disconnect and delete runtime**, reconnect to a GPU (prefer A100/L4), then run
Phase B top-to-bottom. Your Phase-A results are safe on Drive and Phase B merges into the
same `summary/comparison.md` table.
---

## Phase B — Env B models + Part 2 (`transformers@main`)

In [ ]:
# Phase B · Setup — Env B (transformers@main) + training deps (peft/trl/accelerate).
import os
REPO = '/content/MiniVLMDocEval'
if not os.path.isdir(REPO):
    !git clone https://github.com/SajjadPSavoji/MiniVLMDocEval.git $REPO
%cd $REPO
!git pull --quiet && bash setup.sh --bleeding-edge --train

In [ ]:
# Drive — persist all outputs under OUT_DIR (survives disconnects; Phase A & B merge here).
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/MiniVLMDocEval/outputs'
os.makedirs(OUT_DIR, exist_ok=True); print('OUT_DIR ->', OUT_DIR)

In [ ]:
# Phase B · Full eval — Qwen3.5-0.8B + LFM2.5-VL-450M (merges into the same comparison table).
!python scripts/run_eval.py --out $OUT_DIR --n 1000 --fp16 --models Qwen3.5-0.8B LFM2.5-VL-450M

### Part 2 — TableVQA improvement study (Qwen3.5-0.8B)
Diagnose the gap → fine-tune under a **pre-registered no-regression gate** → report faithfully.

In [ ]:
# Part 2 · E0 — baseline TableVQA sub-domain diagnosis (no GPU). VWTQ/VWTQ-Syn are the floor.
!python scripts/tablevqa_subdomain_report.py --out $OUT_DIR --models Qwen3.5-0.8B

In [ ]:
# Part 2 · De-risk — confirm the LoRA adapter attaches to Qwen3.5 BEFORE building data/training.
!python scripts/train_lora_qwen.py --smoke

In [ ]:
# Part 2 · Build the VWTQ-focused training mixture on LOCAL disk (fast; regenerable).
!python scripts/build_table_sft.py --data-dir /content/table_sft --n-total 7000

In [ ]:
# Part 2 · bf16 LoRA fine-tune (1 epoch, vision frozen). Adapter saved to Drive.
!python scripts/train_lora_qwen.py --out $OUT_DIR \
    --data /content/table_sft/train.jsonl --run-name table_lora_v1 --epochs 1

In [ ]:
# Part 2 · Re-evaluate the tuned model on all 5 benchmarks (same seed-42 subsets).
!MVDE_LORA_ADAPTER=$OUT_DIR/lora_adapters/table_lora_v1 \
    python scripts/run_eval.py --out $OUT_DIR --n 1000 --fp16 --models Qwen3.5-0.8B-TableLoRA

In [ ]:
# Part 2 · Verdict — sub-domain before/after + the no-regression gate.
#           (our run FAILS the gate — a catastrophic regression, reported faithfully in Sec 8.)
!python scripts/tablevqa_subdomain_report.py --out $OUT_DIR --models Qwen3.5-0.8B Qwen3.5-0.8B-TableLoRA
!python scripts/regression_gate.py --out $OUT_DIR

In [ ]:
# Part 2 · Export qualitative examples for the report appendix (representative + VWTQ failures).
!python scripts/export_examples.py --out $OUT_DIR --model Qwen3.5-0.8B

### Optional — publish the training dataset to the HuggingFace Hub
Not needed to reproduce results. Fill your **write** token + username, then run.

In [ ]:
# Optional · HF credentials (SECRET — paste at runtime, never commit).
import os
os.environ['HF_TOKEN'] = 'hf_REPLACE_ME'   # your HF *write* token
HF_USERNAME = 'REPLACE_ME'
assert os.environ['HF_TOKEN'].startswith('hf_') and os.environ['HF_TOKEN'] != 'hf_REPLACE_ME'

In [ ]:
# Optional · push dataset -> <username>/minivlm-tablevqa-sft (add --private to keep it private).
!python scripts/push_dataset_hf.py --data-dir /content/table_sft \
    --repo-id $HF_USERNAME/minivlm-tablevqa-sft

In [ ]:
# Optional · push the trained LoRA ADAPTER to <username>/qwen3.5-0.8b-tablevqa-lora.
#            Honest card: this is the experimental adapter that FAILED the gate (Sec 8).
!python scripts/push_model_hf.py \
    --adapter-dir $OUT_DIR/lora_adapters/table_lora_v1 \
    --repo-id $HF_USERNAME/qwen3.5-0.8b-tablevqa-lora

## Phase C — figures for the report (either env)
Regenerates the plots from the results tree. The report PDF itself is compiled locally
(`cd technical_report && latexmk -pdf report.tex`).

In [ ]:
# Phase C · Regenerate report figures (Pareto, per-benchmark, sub-domains, example gallery).
!python scripts/make_report_figures.py --data-dir $OUT_DIR
!python scripts/make_examples_figure.py --src $OUT_DIR/examples